In [26]:
import torch
import torchvision

from torch import nn
from torchvision import datasets
from torchvision.transforms import ToTensor # As the name suggests

import matplotlib.pyplot as plt

In [26]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [27]:
trainData = datasets.FashionMNIST(root='data', train=True, download=True, transform=ToTensor(), target_transform=None)
# Save to data, just download the train set, and convert image to PIL format for later conversion to tensor and no transforms to the labels

In [28]:
testData = datasets.FashionMNIST(root='data', train=False, download=True, transform=ToTensor())

This is a 28 x 28 image set and ToTensor() transform converts it into a tensor of 1x28x28. So, basically we have a 784 features for 10 labels

In [29]:
print(type(trainData))

<class 'torchvision.datasets.mnist.FashionMNIST'>


In [30]:
image, label = trainData[0]

In [31]:
print(image, label)
# Returning the first sameple. I mean its not flat, but you undertand that it is 

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0510,
          0.2863, 0.0000, 0.0000, 0.0039, 0.0157, 0.0000,

In [32]:
# For now use NCHW, its standard practice, ig YOLO uses HWNC

In [33]:
classNames = trainData.classes
print(classNames)

['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


For starters, just know that if you create a dataset instance using `torch.utils.data.Dataset`, you can wrap it around an iterable called dataloader present in `torch.utils.data.DataLoader`

What it does it turn the iterables into small batches. This gives rise to the same name hyperparameter

In [34]:
from torch.utils.data import DataLoader

BATCHSIZE = 32

trainDataLoader = DataLoader(trainData, batch_size=BATCHSIZE, shuffle=True)
#Shuffle data every batch

In [35]:
testDataLoader = DataLoader(testData, batch_size=BATCHSIZE, shuffle=False)

In [36]:
print(f"Of the total training dataset, there are {len(trainDataLoader)} of {BATCHSIZE} batches")

Of the total training dataset, there are 1875 of 32 batches


In [37]:
print(f"Of the total training dataset, there are {len(testDataLoader)} of {BATCHSIZE} batches.")

Of the total training dataset, there are 313 of 32 batches.


In [38]:
trainFeaturesBatch, trainLabelsBatch = next(iter(trainDataLoader))
print(trainFeaturesBatch.shape, trainLabelsBatch.shape)

torch.Size([32, 1, 28, 28]) torch.Size([32])


In [39]:
flattenModel = nn.Flatten()
x = trainFeaturesBatch[0]

output = flattenModel(x)
print(f"Shape of x before flattening: {x.shape}")
print(f"Shape of x after flattening: {output.shape}")

Shape of x before flattening: torch.Size([1, 28, 28])
Shape of x after flattening: torch.Size([1, 784])


In [40]:
class FashionMNISTModel(nn.Module):
    def __init__(self, inputShape: int, outputShape: int, hiddenUnits: int):
        super().__init__()
        self.linearStack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=inputShape, out_features=hiddenUnits),
            nn.Linear(in_features=hiddenUnits, out_features=outputShape)
        )
    def forward(self, x):
        return self.linearStack(x)

In [41]:
model = FashionMNISTModel(inputShape=784, hiddenUnits=10, outputShape=len(classNames))
model.to("cpu")

FashionMNISTModel(
  (linearStack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=10, bias=True)
    (2): Linear(in_features=10, out_features=10, bias=True)
  )
)

In [42]:
def accuracyFunction(yTrue, yPred):
    correct = torch.eq(yTrue, yPred).sum().item()
    acc = (correct/len(yPred)) * 100
    return acc

In [43]:
lossFunction = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(lr=0.01, params=model.parameters())

In [44]:
from tqdm.auto import tqdm # Progressbar lib

In [45]:
torch.manual_seed(69)
epochs = 3

In [46]:
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n----------")
    trainLoss = 0
    for batch, (X,y) in enumerate(trainDataLoader):
        model.train()
        yPred = model(X)
        loss = lossFunction(yPred, y)

        trainLoss += loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 400 == 0:
            print(f"Looked at {batch * len(X)}/{len(trainDataLoader.dataset)} samples")

    trainLoss /= len(trainDataLoader)

    testLoss, testAcc = 0, 0
    model.eval()

    with torch.inference_mode():
        for X, y in testDataLoader:
            testPred = model(X)
            testLoss += lossFunction(testPred, y)
            testAcc += accuracyFunction(yTrue=y, yPred=testPred.argmax(dim=1))
        
        testLoss /= len(testDataLoader)
        testAcc /= len(testDataLoader)

    print(f"\nTrain Loss: {trainLoss:.5f} || Test Loss: {testLoss: 5f} || Test Accuracy: {testAcc:.2f}")

  0%|                                                                                                                         | 0/3 [00:00<?, ?it/s]

Epoch: 0
----------
Looked at 0/60000 samples
Looked at 12800/60000 samples
Looked at 25600/60000 samples
Looked at 38400/60000 samples
Looked at 51200/60000 samples


 33%|█████████████████████████████████████▋                                                                           | 1/3 [00:11<00:23, 11.88s/it]


Train Loss: 0.89519 || Test Loss:  0.640602 || Test Accuracy: 77.21
Epoch: 1
----------
Looked at 0/60000 samples
Looked at 12800/60000 samples
Looked at 25600/60000 samples
Looked at 38400/60000 samples
Looked at 51200/60000 samples


 67%|███████████████████████████████████████████████████████████████████████████▎                                     | 2/3 [00:22<00:11, 11.28s/it]


Train Loss: 0.56564 || Test Loss:  0.557882 || Test Accuracy: 80.56
Epoch: 2
----------
Looked at 0/60000 samples
Looked at 12800/60000 samples
Looked at 25600/60000 samples
Looked at 38400/60000 samples
Looked at 51200/60000 samples


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:33<00:00, 11.19s/it]


Train Loss: 0.50852 || Test Loss:  0.515649 || Test Accuracy: 82.05


In [47]:
def evalMode(model: torch.nn.Module, dataLoader: torch.utils.data.DataLoader, lossFunction: torch.nn.Module, accuracyFunction):
    loss, acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for X,y in dataLoader:
            yPred = model(X)
            loss += lossFunction(yPred, y)
            acc += accuracyFunction(yTrue = y, yPred=yPred.argmax(dim = 1))
        loss /= len(dataLoader)
        acc /= len(dataLoader)

    return {"modelName": model.__class__.__name__, "modelLoss": loss.item(), "modelAcc": acc}


modelResults = evalMode(model=model, dataLoader=testDataLoader, lossFunction=lossFunction, accuracyFunction=accuracyFunction)
print(modelResults)

{'modelName': 'FashionMNISTModel', 'modelLoss': 0.515648603439331, 'modelAcc': 82.04872204472844}


$\text{Let's make the model non linear in the hopes of boosting the accuracy and reducing the training loss even more(close to zero)}$

In [74]:
class FashionMNISTV1(nn.Module):
    def __init__(self, inputShape:int, hiddenUnits:int, outputShape:int):
        super().__init__()
        self.LinearStack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=inputShape, out_features=hiddenUnits),
            nn.ReLU(),
            nn.Linear(in_features=hiddenUnits, out_features=outputShape),
            #nn.ReLU()
        )
    def forward(self, x):
        return self.LinearStack(x)

In [75]:
torch.manual_seed(42)

model1 = FashionMNISTV1(inputShape=784, hiddenUnits=128, outputShape=len(classNames)).to("cpu")

In [76]:
optimizer = torch.optim.SGD(params=model1.parameters(), lr = 0.01)

In [77]:
def trainStep(model:torch.nn.Module, dataLoader:torch.utils.data.DataLoader, 
              lossFunction:torch.nn.Module, optimizer:torch.optim.Optimizer, accuracyFunction,
             device:torch.device="cpu"):
    trainLoss, trainAcc = 0, 0
    model.to(device)
    for batch, (X,y) in enumerate(dataLoader):
        X, y = X.to(device), y.to(device)
        yPred = model(X)

        loss = lossFunction(yPred, y)
        trainLoss += loss
        trainAcc += accuracyFunction(yTrue=y, yPred=yPred.argmax(dim=1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    trainLoss /= len(dataLoader)
    trainAcc /= len(dataLoader)
    print(f"Train Loss: {trainLoss} || Train Accuracy: {trainAcc} \n")

def testStep(model:torch.nn.Module, dataLoader:torch.utils.data.DataLoader, lossFunction:torch.nn.Module, accuracyFunction, device:torch.device="cpu"):
    testLoss, testAcc = 0, 0
    model.to(device)
    model.eval()
    with torch.inference_mode():
        for X, y in dataLoader:
            X, y = X.to(device), y.to(device)

            testPred = model(X)
            testLoss += lossFunction(testPred, y)
            testAcc += accuracyFunction(yTrue=y, yPred=testPred.argmax(dim=1))

        testLoss /= len(dataLoader)
        testAcc /= len(dataLoader)

        print(f"Test Loss: {testLoss} || Test Accuracy: {testAcc} \n")
    

In [78]:
torch.manual_seed(42)

In [79]:
epochs = 3
for epoch in tqdm(range(epochs)):
    print(f"Epoch:{epoch}\n----------")
    trainStep(model=model1, dataLoader=trainDataLoader, lossFunction=lossFunction, accuracyFunction=accuracyFunction, optimizer=optimizer)
    testStep(model=model1, dataLoader=testDataLoader, lossFunction=lossFunction, accuracyFunction=accuracyFunction)

  0%|                                                                                                                         | 0/3 [00:00<?, ?it/s]

Epoch:0
----------
Train Loss: 0.9073857665061951 || Train Accuracy: 70.46 



 33%|█████████████████████████████████████▋                                                                           | 1/3 [00:10<00:21, 10.89s/it]

Test Loss: 0.6349039673805237 || Test Accuracy: 78.40455271565496 

Epoch:1
----------
Train Loss: 0.5607700347900391 || Train Accuracy: 81.03666666666666 



 67%|███████████████████████████████████████████████████████████████████████████▎                                     | 2/3 [00:21<00:10, 10.69s/it]

Test Loss: 0.5404857397079468 || Test Accuracy: 81.29992012779553 

Epoch:2
----------
Train Loss: 0.49803948402404785 || Train Accuracy: 82.84666666666666 



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:31<00:00, 10.64s/it]

Test Loss: 0.5049964785575867 || Test Accuracy: 82.24840255591054 

